# 8 Rainhas — Formulação de Estados Completos

Duas abordagens sobre a mesma formulação: busca clássica em espaço de estados com poda e busca local por subida de encosta, com discussão dos máximos locais.

**Técnica:** BFS com poda e hill climbing  
**Referência:** Russell & Norvig, *Inteligência Artificial* (AIMA)  
**Contexto:** disciplina de Inteligência Artificial — Análise e Desenvolvimento de Sistemas, FATEC Taubaté

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/devcauas/agentes-ia/blob/main/notebooks/07-8-rainhas-estados-completos.ipynb)


In [3]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║          PROBLEMA DAS 8 RAINHAS — FORMULAÇÃO DE ESTADOS COMPLETOS           ║
║                    Disciplina: Inteligência Artificial                       ║
║         Referência: Russell & Norvig — Inteligência Artificial (Cap. 3-4)   ║
╚══════════════════════════════════════════════════════════════════════════════╝

Este projeto implementa duas abordagens:
  - Parte 1: Busca Clássica em Espaço de Estados (BFS com poda)
  - Parte 2: Busca Local com Hill Climbing (Subida de Encosta)

CONCEITOS FUNDAMENTAIS (conforme aula):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
► Espaço de Estados: Conjunto de todos os estados acessíveis a partir do
  estado inicial por qualquer sequência de ações. Pode ser visto como um
  grafo onde nós são estados e arcos são ações.

► Árvore de Busca: Estrutura gerada durante a exploração do espaço de
  estados. NÃO é equivalente ao espaço de estados! Um mesmo estado pode
  aparecer em múltiplos nós da árvore (via caminhos diferentes).
  A árvore de busca "cresce sobre" o grafo do espaço de estados.

► Estados Completos: Cada estado já contém TODAS as 8 rainhas no tabuleiro,
  diferente da formulação incremental que adiciona uma rainha por vez.
"""

import random
import time
import math
from collections import deque
from typing import Optional, List, Tuple


# ══════════════════════════════════════════════════════════════════════════════
# UTILITÁRIOS GERAIS
# ══════════════════════════════════════════════════════════════════════════════

def renderizar_tabuleiro_classico(estado: List[Tuple[int, int]], n: int = 8) -> str:
    """
    Renderiza o tabuleiro da formulação clássica.
    Estado = lista de tuplas (linha, coluna).
    """
    grid = [['.' for _ in range(n)] for _ in range(n)]
    for (linha, coluna) in estado:
        grid[linha][coluna] = 'Q'
    return '\n'.join(' '.join(row) for row in grid)


def renderizar_tabuleiro_local(estado: List[int], n: int = 8) -> str:
    """
    Renderiza o tabuleiro da busca local.
    Estado = vetor onde índice=coluna, valor=linha.
    """
    grid = [['.' for _ in range(n)] for _ in range(n)]
    for coluna, linha in enumerate(estado):
        grid[linha][coluna] = 'Q'
    return '\n'.join(' '.join(row) for row in grid)


def separador(char: str = '═', largura: int = 70) -> str:
    return char * largura


def titulo(texto: str, char: str = '═') -> str:
    pad = max(0, 70 - len(texto) - 4)
    esq = pad // 2
    dir_ = pad - esq
    return f"{'═'*esq}  {texto}  {'═'*dir_}"


# ══════════════════════════════════════════════════════════════════════════════
# PARTE 1 — FORMULAÇÃO CLÁSSICA DE ESTADOS COMPLETOS
# ══════════════════════════════════════════════════════════════════════════════

class No:
    """
    Estrutura de Nó da Árvore de Busca (conforme definição clássica de IA).

    Cada nó contém:
      - estado:         configuração atual do tabuleiro
      - pai:            nó que gerou este nó (None se for a raiz)
      - acao:           ação aplicada ao pai para chegar aqui
      - custo_caminho:  g(n) — custo acumulado da raiz até aqui
      - profundidade:   nível do nó na árvore de busca

    ─────────────────────────────────────────────
    IMPORTANTE: Nó ≠ Estado!
      Estado = configuração física do tabuleiro
      Nó     = estrutura da árvore (contém o estado + metadados de busca)
    ─────────────────────────────────────────────
    """

    def __init__(
        self,
        estado: List[Tuple[int, int]],
        pai: Optional['No'] = None,
        acao: Optional[str] = None,
        custo_caminho: int = 0,
        profundidade: int = 0
    ):
        self.estado = estado
        self.pai = pai
        self.acao = acao
        self.custo_caminho = custo_caminho
        self.profundidade = profundidade

    def __repr__(self):
        return (f"No(rainhas={len(self.estado)}, "
                f"prof={self.profundidade}, custo={self.custo_caminho})")


class ProblemaOitoRainhasClassico:
    """
    Formulação Clássica do Problema das 8 Rainhas.

    ╔═══════════════════════════════════════════════════════════╗
    ║              DEFINIÇÃO FORMAL DO PROBLEMA                 ║
    ╠═══════════════════════════════════════════════════════════╣
    ║ Estados:         Qualquer disposição de 0 a 8 rainhas    ║
    ║ Estado inicial:  Tabuleiro vazio (nenhuma rainha)         ║
    ║ Função sucessor: Adicionar rainha em qualquer quadrado   ║
    ║ Teste objetivo:  8 rainhas, nenhuma se atacando          ║
    ╚═══════════════════════════════════════════════════════════╝

    REPRESENTAÇÃO INTERNA:
      Lista de tuplas (linha, coluna).
      Estado vazio = []
      Exemplo com 2 rainhas: [(0,0), (3,5)]

    ANÁLISE DE COMPLEXIDADE:
    ─────────────────────────
    A função sucessor adiciona uma rainha em qualquer quadrado vazio:
      Nível 0 → 64 escolhas
      Nível 1 → 63 escolhas
      ...
      Nível 7 → 57 escolhas

    Total de sequências = 64 × 63 × 62 × 61 × 60 × 59 × 58 × 57
                        = 178.462.987.637.760
                        ≈ 1,8 × 10¹⁴ sequências

    Isso torna a busca exaustiva COMPUTACIONALMENTE INVIÁVEL.
    BFS precisaria armazenar todos os nós da borda em memória — impossível.
    """

    def __init__(self, n: int = 8):
        self.n = n

    def estado_inicial(self) -> List[Tuple[int, int]]:
        """Retorna o estado inicial: tabuleiro completamente vazio."""
        return []

    def gerar_sucessores(
        self, estado: List[Tuple[int, int]]
    ) -> List[Tuple[List[Tuple[int, int]], str]]:
        """
        Função Sucessor da Formulação Clássica.

        Para um estado com k rainhas, gera todos os estados com k+1 rainhas,
        posicionando a próxima rainha em qualquer quadrado vazio.

        Fator de ramificação (b) por nível:
          - Nível 0 (0 rainhas): b = 64
          - Nível 1 (1 rainha):  b = 63
          - ...
          - Nível 7 (7 rainhas): b = 57

        Retorna lista de (novo_estado, descrição_da_ação).
        """
        ocupadas = set(estado)
        sucessores = []
        for linha in range(self.n):
            for coluna in range(self.n):
                if (linha, coluna) not in ocupadas:
                    novo_estado = estado + [(linha, coluna)]
                    acao = f"Rainha em ({linha},{coluna})"
                    sucessores.append((novo_estado, acao))
        return sucessores

    def verificar_ataque(
        self, pos1: Tuple[int, int], pos2: Tuple[int, int]
    ) -> bool:
        """
        Verifica se duas rainhas se atacam mutuamente.

        Casos de ataque:
          1. Mesma linha:    l1 == l2
          2. Mesma coluna:   c1 == c2
          3. Diagonal:       |l1-l2| == |c1-c2|
             (cobre diagonal principal e secundária)

        Retorna True se houver ataque, False caso contrário.
        """
        l1, c1 = pos1
        l2, c2 = pos2
        if l1 == l2:              # mesma linha
            return True
        if c1 == c2:              # mesma coluna
            return True
        if abs(l1-l2) == abs(c1-c2):  # diagonal
            return True
        return False

    def teste_objetivo(self, estado: List[Tuple[int, int]]) -> bool:
        """
        Teste de Objetivo: verifica se o estado é a solução.

        Condições necessárias e suficientes:
          1. Exatamente 8 rainhas no tabuleiro
          2. Nenhum par de rainhas se ataca

        Complexidade: O(n²) — analisa todos os pares (n×(n-1)/2 pares)
        """
        if len(estado) != self.n:
            return False
        for i in range(len(estado)):
            for j in range(i + 1, len(estado)):
                if self.verificar_ataque(estado[i], estado[j]):
                    return False
        return True

    def extrair_solucao(self, no: 'No') -> List[Tuple]:
        """
        Extrai o caminho da solução seguindo os ponteiros PAI de volta à raiz.
        Retorna a sequência de (estado, ação) do estado inicial até a solução.
        """
        caminho = []
        atual = no
        while atual is not None:
            caminho.append((atual.estado, atual.acao))
            atual = atual.pai
        caminho.reverse()
        return caminho

    def sequencias_possiveis(self) -> int:
        """Calcula o número de sequências possíveis (64 × 63 × ... × 57)."""
        total = 1
        for i in range(self.n * self.n, self.n * self.n - self.n, -1):
            total *= i
        return total


def busca_largura_com_poda(
    problema: ProblemaOitoRainhasClassico,
    limite_nos: int = 5000
) -> Tuple[Optional[No], int, int, float]:
    """
    Busca em Largura (BFS) com poda de estados inválidos.

    BFS PURO (sem poda):
    ──────────────────────
    • Fila FIFO — expande nós nível por nível
    • Garante solução de menor profundidade
    • Complexidade de tempo:  O(b^d) — INVIÁVEL sem poda
    • Complexidade de espaço: O(b^d) — mantém toda a borda em memória

    BFS COM PODA (implementado aqui):
    ───────────────────────────────────
    • Gera sucessores apenas em colunas ainda não usadas (uma rainha por col.)
    • Rejeita estados com conflitos antes de continuar a busca
    • Reduz drasticamente o espaço explorado

    Mesmo com poda, aplicamos um limite para demonstração didática,
    já que o espaço ainda é enorme (1,8 × 10¹⁴ sem restrições).

    Retorna: (nó_solução, nós_gerados, nós_expandidos, tempo)
    """
    inicio = time.time()
    nos_gerados = 0
    nos_expandidos = 0

    estado_inicial = problema.estado_inicial()
    no_raiz = No(
        estado=estado_inicial,
        pai=None,
        acao="Estado inicial (tabuleiro vazio)",
        custo_caminho=0,
        profundidade=0
    )

    borda = deque([no_raiz])
    nos_gerados += 1

    while borda:
        if nos_expandidos >= limite_nos:
            return None, nos_gerados, nos_expandidos, time.time() - inicio

        no_atual = borda.popleft()
        nos_expandidos += 1

        # Teste de objetivo
        if problema.teste_objetivo(no_atual.estado):
            return no_atual, nos_gerados, nos_expandidos, time.time() - inicio

        # Não expande se já tem 8 rainhas (sem solução encontrada)
        if len(no_atual.estado) >= problema.n:
            continue

        # ── Poda inteligente: gerar sucessores apenas em colunas livres ──
        # Determina quais colunas já estão ocupadas
        colunas_usadas = {c for (_, c) in no_atual.estado}
        # Próxima coluna a preencher (garante uma rainha por coluna)
        proxima_coluna = len(no_atual.estado)  # coluna 0, 1, 2, ...

        # Tenta cada linha nessa coluna
        for linha in range(problema.n):
            novo_estado = no_atual.estado + [(linha, proxima_coluna)]

            # Poda: verifica se a nova rainha conflita com as existentes
            conflito = False
            for pos_existente in no_atual.estado:
                if problema.verificar_ataque((linha, proxima_coluna), pos_existente):
                    conflito = True
                    break

            if not conflito:
                no_filho = No(
                    estado=novo_estado,
                    pai=no_atual,
                    acao=f"Rainha em ({linha},{proxima_coluna})",
                    custo_caminho=no_atual.custo_caminho + 1,
                    profundidade=no_atual.profundidade + 1
                )
                borda.append(no_filho)
                nos_gerados += 1

    return None, nos_gerados, nos_expandidos, time.time() - inicio


def executar_parte1():
    """
    Executa e exibe a Parte 1: Busca Clássica em Espaço de Estados.
    """
    print()
    print(titulo("PARTE 1 — BUSCA CLÁSSICA EM ESPAÇO DE ESTADOS"))
    print()

    problema = ProblemaOitoRainhasClassico(n=8)
    seq = problema.sequencias_possiveis()

    print("FORMULAÇÃO DO PROBLEMA:")
    print("  • Estados:         Qualquer disposição de 0 a 8 rainhas no tabuleiro")
    print("  • Estado inicial:  Tabuleiro vazio (0 rainhas)")
    print("  • Função sucessor: Adicionar rainha em qualquer quadrado vazio")
    print("  • Teste objetivo:  8 rainhas, nenhuma se atacando")
    print()
    print("ESTRUTURA DO NÓ (árvore de busca):")
    print("  • estado:         posições das rainhas no tabuleiro")
    print("  • pai:            ponteiro para o nó gerador")
    print("  • ação:           qual rainha foi adicionada e onde")
    print("  • custo_caminho:  g(n) = número de movimentos desde a raiz")
    print("  • profundidade:   nível na árvore de busca")
    print()
    print("ESPAÇO DE ESTADOS vs ÁRVORE DE BUSCA:")
    print("  ► O espaço de estados é o GRAFO de todos os estados possíveis.")
    print("  ► A árvore de busca é gerada DURANTE a exploração.")
    print("  ► Um mesmo estado pode aparecer em MÚLTIPLOS nós da árvore")
    print("    (via caminhos distintos), mas representa o mesmo ponto no grafo.")
    print()
    print("ANÁLISE DE COMPLEXIDADE DA FORMULAÇÃO CLÁSSICA:")
    print(f"  Fator de ramificação (b): até 64 por nível")
    print(f"  Profundidade da solução (d): 8")
    print(f"  Sequências possíveis: 64×63×62×61×60×59×58×57")
    print(f"  = {seq:,} ≈ 1,8 × 10¹⁴")
    print()
    print("  Por que isso é inviável?")
    print("  BFS armazena TODA a borda em memória — O(b^d) nós.")
    print("  Com b≈60 e d=8, isso ultrapassa qualquer memória real.")
    print("  DFS evita o problema de memória, mas pode seguir caminhos")
    print("  infinitos sem encontrar solução.")
    print()
    print(separador('─'))
    print("EXECUTANDO BFS COM PODA (uma rainha por coluna + rejeição de conflitos):")
    print(separador('─'))
    print()

    no_solucao, nos_gerados, nos_expandidos, tempo = busca_largura_com_poda(
        problema, limite_nos=5000
    )

    if no_solucao:
        print(f"  ✓ SOLUÇÃO ENCONTRADA!")
        print(f"  • Nós gerados:    {nos_gerados}")
        print(f"  • Nós expandidos: {nos_expandidos}")
        print(f"  • Profundidade:   {no_solucao.profundidade}")
        print(f"  • Custo caminho:  {no_solucao.custo_caminho}")
        print(f"  • Tempo:          {tempo:.4f}s")
        print()
        print("  TABULEIRO SOLUÇÃO (BFS com poda):")
        print()
        for linha in renderizar_tabuleiro_classico(no_solucao.estado).split('\n'):
            print(f"    {linha}")
        print()
        print("  SEQUÊNCIA DE AÇÕES (caminho da raiz até a solução):")
        caminho = problema.extrair_solucao(no_solucao)
        for i, (est, acao) in enumerate(caminho):
            print(f"    Passo {i}: {acao} → {len(est)} rainha(s)")
    else:
        print(f"  ⚠  Limite de nós atingido ({nos_expandidos} expandidos).")
        print(f"  Mesmo com poda, o espaço clássico é muito grande.")
        print(f"  Isso demonstra a inviabilidade da formulação clássica pura.")

    print()
    print("FORMULAÇÃO REDUZIDA (versão incremental eficiente):")
    print("  • Adiciona rainhas apenas em colunas vazias")
    print("  • Rejeita estados com conflitos imediatamente")
    print("  • Espaço reduzido para ≈ 2.057 estados válidos")
    print(f"  • Redução: {seq:,} → ~2.057 estados")
    print(f"  • Fator de redução: ≈ {seq//2057:,}× menos estados!")
    print()


# ══════════════════════════════════════════════════════════════════════════════
# PARTE 2 — BUSCA LOCAL COM HILL CLIMBING (SUBIDA DE ENCOSTA)
# ══════════════════════════════════════════════════════════════════════════════

class ProblemaOitoRainhasLocal:
    """
    Formulação para Busca Local — Estados Completos.

    ╔═══════════════════════════════════════════════════════════╗
    ║         FORMULAÇÃO PARA BUSCA LOCAL                       ║
    ╠═══════════════════════════════════════════════════════════╣
    ║ Estado:     Vetor com 8 rainhas, uma por coluna           ║
    ║ Heurística: h(n) = pares de rainhas que se atacam        ║
    ║ Objetivo:   h(n) = 0                                      ║
    ║ Vizinhos:   8 × 7 = 56 movimentos possíveis              ║
    ╚═══════════════════════════════════════════════════════════╝

    REPRESENTAÇÃO:
    ─────────────
    Vetor onde índice=coluna, valor=linha da rainha.
    Exemplo: [0, 4, 7, 5, 2, 6, 1, 3]
      col 0 → linha 0  (rainha na posição (0,0))
      col 1 → linha 4  (rainha na posição (4,1))
      col 2 → linha 7  (rainha na posição (7,2))
      ...

    Isso GARANTE: nunca há duas rainhas na mesma coluna.

    POR QUE 56 VIZINHOS?
    ────────────────────
    Cada estado tem exatamente 8 rainhas (uma por coluna).
    Para cada coluna (8 colunas), podemos mover a rainha
    para qualquer das outras 7 linhas do tabuleiro:
      8 colunas × 7 posições alternativas = 56 vizinhos
    """

    def __init__(self, n: int = 8):
        self.n = n

    def estado_aleatorio(self) -> List[int]:
        """Gera estado inicial aleatório: uma rainha por coluna em linha aleatória."""
        return [random.randint(0, self.n - 1) for _ in range(self.n)]

    def calcular_heuristica(self, estado: List[int]) -> int:
        """
        Heurística h(n) = número de pares de rainhas que se atacam.

        Verifica:
          • Mesma linha:  estado[i] == estado[j]
          • Diagonal:     |estado[i] - estado[j]| == |i - j|

        Ataques por coluna são IMPOSSÍVEIS nesta representação.

        Objetivo: MINIMIZAR h(n)
        Solução:  h(n) = 0 (nenhum conflito)

        Complexidade: O(n²) — verifica n×(n-1)/2 pares
        """
        conflitos = 0
        for i in range(self.n):
            for j in range(i + 1, self.n):
                if estado[i] == estado[j]:          # mesma linha
                    conflitos += 1
                elif abs(estado[i] - estado[j]) == abs(i - j):  # diagonal
                    conflitos += 1
        return conflitos

    def gerar_vizinhos(
        self, estado: List[int]
    ) -> List[Tuple[List[int], int, int, int]]:
        """
        Gera todos os 56 vizinhos do estado atual.

        Para cada coluna i (0 a 7):
          Para cada linha j ≠ estado[i] (7 opções):
            Cria novo estado movendo rainha da coluna i para linha j

        Retorna lista de (novo_estado, coluna, nova_linha, heuristica).

        Por que 56?
          8 colunas × 7 linhas alternativas = 56 vizinhos (exato)
        """
        vizinhos = []
        for coluna in range(self.n):
            for linha in range(self.n):
                if linha != estado[coluna]:  # não gera o estado atual
                    novo_estado = estado[:]  # cópia
                    novo_estado[coluna] = linha
                    h = self.calcular_heuristica(novo_estado)
                    vizinhos.append((novo_estado, coluna, linha, h))
        return vizinhos


def hill_climbing(
    problema: ProblemaOitoRainhasLocal,
    estado_inicial: Optional[List[int]] = None,
    max_sideways: int = 0
) -> Tuple[List[int], int, int, List[int]]:
    """
    Hill Climbing (Subida de Encosta) — Algoritmo de Busca Local.

    COMO FUNCIONA:
    ─────────────
    1. Inicia com estado aleatório (ou fornecido)
    2. Calcula h(n) do estado atual
    3. Gera todos os 56 vizinhos
    4. Seleciona o vizinho com menor h(n)
    5. Se h(vizinho) < h(atual): move para o vizinho e repete
    6. Para quando: h=0 (solução) ou nenhum vizinho é melhor (ótimo local)

    PROBLEMAS DO HILL CLIMBING:
    ────────────────────────────
    • Mínimos locais: estado onde todos os vizinhos têm h maior,
      mas o estado não é solução (h > 0). O algoritmo fica preso.
    • Platôs: região onde todos os vizinhos têm o mesmo h.
      O algoritmo não sabe para onde ir.
    • Cristas (ridges): sequência de mínimos locais em que o algoritmo
      não consegue subir sem primeiro descer.

    MELHORIA — sideways moves:
      Permite mover para vizinhos com MESMO h (não só menores).
      Ajuda a escapar de platôs, mas pode causar loops.
      Controlado por max_sideways.

    Retorna: (estado_final, h_final, iterações, historico_h)
    """
    estado = estado_inicial if estado_inicial else problema.estado_aleatorio()
    h_atual = problema.calcular_heuristica(estado)
    iteracoes = 0
    sideways_count = 0
    historico_h = [h_atual]

    while True:
        iteracoes += 1
        vizinhos = problema.gerar_vizinhos(estado)

        # Ordena vizinhos por heurística (menor é melhor)
        vizinhos.sort(key=lambda x: x[3])
        melhor_h = vizinhos[0][3]

        if melhor_h < h_atual:
            # Move para o melhor vizinho
            # Escolha aleatória entre empatados para diversidade
            melhores = [v for v in vizinhos if v[3] == melhor_h]
            escolhido = random.choice(melhores)
            estado = escolhido[0]
            h_atual = melhor_h
            sideways_count = 0
            historico_h.append(h_atual)

        elif melhor_h == h_atual and sideways_count < max_sideways:
            # Sideways move: permite mover em platô
            melhores = [v for v in vizinhos if v[3] == melhor_h]
            escolhido = random.choice(melhores)
            estado = escolhido[0]
            sideways_count += 1
            historico_h.append(h_atual)

        else:
            # Ótimo local (ou platô esgotado): para aqui
            break

        if h_atual == 0:
            break  # SOLUÇÃO ENCONTRADA!

    return estado, h_atual, iteracoes, historico_h


def hill_climbing_random_restart(
    problema: ProblemaOitoRainhasLocal,
    max_restarts: int = 100,
    max_sideways: int = 5
) -> Tuple[Optional[List[int]], int, int, int, List[int]]:
    """
    Hill Climbing com Random Restart.

    Resolve o problema dos mínimos locais: quando o Hill Climbing
    fica preso, reinicia com um estado inicial aleatório diferente.

    Com restarts suficientes, tem probabilidade alta de encontrar
    solução para o problema das 8 Rainhas.

    Retorna: (solução, h_final, total_iterações, restarts_usados, historico)
    """
    total_iteracoes = 0
    historico_completo = []

    for restart in range(max_restarts):
        estado, h, iteracoes, hist = hill_climbing(
            problema, max_sideways=max_sideways
        )
        total_iteracoes += iteracoes
        historico_completo.extend(hist)

        if h == 0:
            return estado, h, total_iteracoes, restart + 1, historico_completo

    return None, -1, total_iteracoes, max_restarts, historico_completo


def executar_parte2():
    """
    Executa e exibe a Parte 2: Busca Local com Hill Climbing.
    """
    print()
    print(titulo("PARTE 2 — BUSCA LOCAL: HILL CLIMBING (SUBIDA DE ENCOSTA)"))
    print()

    problema = ProblemaOitoRainhasLocal(n=8)

    print("FORMULAÇÃO PARA BUSCA LOCAL:")
    print("  • Estado:         Vetor de 8 inteiros — índice=coluna, valor=linha")
    print("  • Estado inicial: Aleatório (8 rainhas, uma por coluna)")
    print("  • Vizinhos:       8 × 7 = 56 movimentos (mover rainha na coluna)")
    print("  • Heurística:     h(n) = nº de pares de rainhas que se atacam")
    print("  • Objetivo:       h(n) = 0")
    print()
    print("POR QUE 56 VIZINHOS?")
    print("  Cada coluna tem 1 rainha que pode ir para as outras 7 linhas.")
    print("  8 colunas × 7 posições alternativas = 56 vizinhos por estado.")
    print()

    # ── 2a. Hill Climbing simples ──
    print(separador('─'))
    print("2a. HILL CLIMBING SIMPLES (sem sideways, sem restart):")
    print(separador('─'))
    print()

    inicio = time.time()
    estado_ini = problema.estado_aleatorio()
    print(f"  Estado inicial: {estado_ini}")
    print(f"  h(inicial) = {problema.calcular_heuristica(estado_ini)}")
    print()

    estado_final, h_final, iters, historico = hill_climbing(problema, estado_ini)
    tempo = time.time() - inicio

    print(f"  Estado final:  {estado_final}")
    print(f"  h(final) = {h_final}")
    print(f"  Iterações: {iters}")
    print(f"  Tempo: {tempo:.4f}s")

    if h_final == 0:
        print(f"  ✓ SOLUÇÃO ENCONTRADA NA PRIMEIRA TENTATIVA!")
    else:
        print(f"  ✗ Preso em mínimo local (h={h_final} > 0)")
        print(f"    → Todos os 56 vizinhos têm h ≥ {h_final}")

    print()
    print("  Histórico de h(n) durante a busca:")
    print(f"  {' → '.join(map(str, historico))}")
    print()

    if h_final == 0:
        print("  TABULEIRO SOLUÇÃO:")
        print()
        for linha in renderizar_tabuleiro_local(estado_final).split('\n'):
            print(f"    {linha}")
    print()

    # ── 2b. Hill Climbing com Sideways Moves ──
    print(separador('─'))
    print("2b. HILL CLIMBING COM SIDEWAYS MOVES (max=10):")
    print(separador('─'))
    print()
    print("  Sideways moves permitem mover para vizinhos com MESMO h.")
    print("  Ajuda a escapar de platôs, mas pode causar loops.")
    print()

    inicio = time.time()
    estado_ini2 = problema.estado_aleatorio()
    print(f"  Estado inicial: {estado_ini2}")
    print(f"  h(inicial) = {problema.calcular_heuristica(estado_ini2)}")

    estado_sw, h_sw, iters_sw, hist_sw = hill_climbing(
        problema, estado_ini2, max_sideways=10
    )
    tempo_sw = time.time() - inicio

    print(f"  Estado final:  {estado_sw}")
    print(f"  h(final) = {h_sw}  |  Iterações: {iters_sw}  |  Tempo: {tempo_sw:.4f}s")

    if h_sw == 0:
        print(f"  ✓ SOLUÇÃO COM SIDEWAYS MOVES!")
        print()
        print("  TABULEIRO SOLUÇÃO:")
        print()
        for linha in renderizar_tabuleiro_local(estado_sw).split('\n'):
            print(f"    {linha}")
    else:
        print(f"  ✗ Ainda preso (h={h_sw}) mesmo com sideways.")
    print()

    # ── 2c. Hill Climbing com Random Restart ──
    print(separador('─'))
    print("2c. HILL CLIMBING COM RANDOM RESTART (até 100 reinícios):")
    print(separador('─'))
    print()
    print("  Quando fica preso, reinicia com estado inicial aleatório.")
    print("  Garante encontrar solução com alta probabilidade.")
    print()

    inicio = time.time()
    resultado, h_rr, total_iters, restarts, hist_rr = hill_climbing_random_restart(
        problema, max_restarts=100, max_sideways=5
    )
    tempo_rr = time.time() - inicio

    if resultado and h_rr == 0:
        print(f"  ✓ SOLUÇÃO ENCONTRADA!")
        print(f"  • Restarts necessários: {restarts}")
        print(f"  • Total de iterações:   {total_iters}")
        print(f"  • Tempo total:          {tempo_rr:.4f}s")
        print(f"  • Média por restart:    {total_iters/restarts:.1f} iterações")
        print()
        print("  TABULEIRO SOLUÇÃO FINAL:")
        print()
        for linha in renderizar_tabuleiro_local(resultado).split('\n'):
            print(f"    {linha}")
        print()
        print(f"  Solução vetor: {resultado}")
        print(f"  Verificação h(n) = {problema.calcular_heuristica(resultado)}")
    else:
        print(f"  ✗ Não encontrou solução em {restarts} restarts.")
    print()

    # ── Explicação dos problemas do Hill Climbing ──
    print(separador('─'))
    print("PROBLEMAS DO HILL CLIMBING (por que pode falhar):")
    print(separador('─'))
    print()
    print("  ► MÍNIMOS LOCAIS:")
    print("    Estado onde h(n) > 0, mas todos os 56 vizinhos têm h maior.")
    print("    O algoritmo interpreta como 'topo da colina', mas não é solução.")
    print("    O algoritmo para sem encontrar h=0.")
    print()
    print("  ► PLATÔS:")
    print("    Região onde vários estados vizinhos têm o mesmo h(n).")
    print("    O algoritmo não sabe para qual direção avançar.")
    print("    Sideways moves ajudam, mas podem criar loops infinitos.")
    print()
    print("  ► CRISTAS (RIDGES):")
    print("    Sequência de mínimos locais formando uma 'cordilheira'.")
    print("    Para ir para frente, é preciso primeiro 'descer',")
    print("    o que Hill Climbing clássico não faz.")
    print()
    print("  Estatística empírica para as 8 Rainhas:")
    print("  Hill Climbing simples encontra solução em ≈ 14% dos casos.")
    print("  Com random restart: encontra solução em poucos segundos.")
    print()

    return resultado, h_rr


# ══════════════════════════════════════════════════════════════════════════════
# SEÇÃO 3 — COMPARAÇÃO ENTRE FORMULAÇÕES
# ══════════════════════════════════════════════════════════════════════════════

def exibir_comparacao():
    """
    Exibe comparação crítica entre as três formulações.
    """
    print()
    print(titulo("SEÇÃO 3 — COMPARAÇÃO ENTRE FORMULAÇÕES"))
    print()

    seq_classica = 64 * 63 * 62 * 61 * 60 * 59 * 58 * 57
    estados_reduzidos = 2057

    print("┌─────────────────────────────────────────────────────────────────┐")
    print("│               FORMULAÇÃO CLÁSSICA (completa)                    │")
    print("├─────────────────────────────────────────────────────────────────┤")
    print("│  • Estado: qualquer disposição de 0-8 rainhas                   │")
    print("│  • Sucessor: adicionar rainha em QUALQUER quadrado vazio        │")
    print(f"│  • Espaço: ≈ 1,8 × 10¹⁴ sequências ({seq_classica:,})  │")
    print("│  • BFS/DFS: inviável sem poda                                   │")
    print("│  • Não há heurística — busca exaustiva                         │")
    print("└─────────────────────────────────────────────────────────────────┘")
    print()
    print("┌─────────────────────────────────────────────────────────────────┐")
    print("│          FORMULAÇÃO INCREMENTAL REDUZIDA (com poda)             │")
    print("├─────────────────────────────────────────────────────────────────┤")
    print("│  • Estado: rainhas adicionadas coluna a coluna                  │")
    print("│  • Sucessor: apenas em colunas livres + sem conflitos           │")
    print(f"│  • Espaço reduzido: ≈ {estados_reduzidos} estados válidos                    │")
    print(f"│  • Redução: {seq_classica:,} → {estados_reduzidos}            │")
    print(f"│  • Fator de melhoria: ≈ {seq_classica//estados_reduzidos:,}×                │")
    print("│  • BFS/DFS viável — encontra solução rapidamente                │")
    print("└─────────────────────────────────────────────────────────────────┘")
    print()
    print("┌─────────────────────────────────────────────────────────────────┐")
    print("│           FORMULAÇÃO LOCAL (Hill Climbing)                      │")
    print("├─────────────────────────────────────────────────────────────────┤")
    print("│  • Estado: 8 rainhas completas (uma por coluna)                 │")
    print("│  • Sem caminho, sem nó pai — apenas estado atual                │")
    print("│  • Heurística h(n) = pares atacantes (minimizar)               │")
    print("│  • 56 vizinhos por estado (8 colunas × 7 linhas)               │")
    print("│  • Não garante solução ótima (pode ficar em mínimo local)      │")
    print("│  • Com random restart: encontra solução em segundos             │")
    print("│  • Complexidade de espaço: O(1) — só armazena estado atual     │")
    print("└─────────────────────────────────────────────────────────────────┘")
    print()
    print("DIFERENÇA FUNDAMENTAL: BFS/DFS × HILL CLIMBING:")
    print()
    print("  BFS e DFS mantêm uma ÁRVORE DE BUSCA explícita:")
    print("    • Armazenam os nós visitados e a borda de exploração")
    print("    • Podem reconstruir o CAMINHO da solução (sequência de ações)")
    print("    • Garantem completeza (BFS garante solução ótima)")
    print("    • Alto custo de memória: O(b^d)")
    print()
    print("  Hill Climbing é uma busca LOCAL sem memória:")
    print("    • Mantém APENAS o estado atual (sem árvore, sem caminho)")
    print("    • Não pode reconstruir como chegou à solução")
    print("    • Não garante completeza nem otimização")
    print("    • Custo de memória: O(1) — extremamente eficiente")
    print("    • Adequado quando o ESTADO FINAL importa, não o caminho")
    print()
    print("  Para as 8 Rainhas, o ESTADO FINAL é o que importa!")
    print("  Por isso, Hill Climbing + Random Restart é a abordagem prática.")
    print()


# ══════════════════════════════════════════════════════════════════════════════
# EXECUÇÃO PRINCIPAL
# ══════════════════════════════════════════════════════════════════════════════

def main():
    random.seed(42)  # Para reprodutibilidade dos exemplos

    print()
    print(separador())
    print(titulo("PROBLEMA DAS 8 RAINHAS — INTELIGÊNCIA ARTIFICIAL"))
    print(titulo("Formulação de Estados Completos", '─'))
    print(separador())

    # ── Parte 1: Busca Clássica ──
    executar_parte1()

    # ── Parte 2: Busca Local ──
    solucao, h = executar_parte2()

    # ── Seção 3: Comparação ──
    exibir_comparacao()

    # ── Resumo Final ──
    print(separador())
    print(titulo("RESUMO FINAL"))
    print(separador())
    print()
    print("  Formulação Clássica (BFS puro):")
    print("    → 1,8 × 10¹⁴ sequências | Inviável sem poda")
    print()
    print("  BFS com Poda (formulação incremental):")
    print("    → ≈ 2.057 estados | Rápido e completo")
    print()
    print("  Hill Climbing com Random Restart:")
    print("    → Encontra solução em poucos segundos")
    print("    → Melhor abordagem prática para o problema")
    print()
    if solucao and h == 0:
        print("  SOLUÇÃO FINAL VERIFICADA:")
        print()
        for linha in renderizar_tabuleiro_local(solucao).split('\n'):
            print(f"    {linha}")
        print()
        print(f"  Vetor: {solucao}")
        p = ProblemaOitoRainhasLocal()
        print(f"  h(n) = {p.calcular_heuristica(solucao)} ✓ (nenhum conflito)")
    print()
    print(separador())


if __name__ == "__main__":
    main()


══════════════════════════════════════════════════════════════════════
═════════  PROBLEMA DAS 8 RAINHAS — INTELIGÊNCIA ARTIFICIAL  ═════════
═════════════════  Formulação de Estados Completos  ══════════════════
══════════════════════════════════════════════════════════════════════

══════════  PARTE 1 — BUSCA CLÁSSICA EM ESPAÇO DE ESTADOS  ═══════════

FORMULAÇÃO DO PROBLEMA:
  • Estados:         Qualquer disposição de 0 a 8 rainhas no tabuleiro
  • Estado inicial:  Tabuleiro vazio (0 rainhas)
  • Função sucessor: Adicionar rainha em qualquer quadrado vazio
  • Teste objetivo:  8 rainhas, nenhuma se atacando

ESTRUTURA DO NÓ (árvore de busca):
  • estado:         posições das rainhas no tabuleiro
  • pai:            ponteiro para o nó gerador
  • ação:           qual rainha foi adicionada e onde
  • custo_caminho:  g(n) = número de movimentos desde a raiz
  • profundidade:   nível na árvore de busca

ESPAÇO DE ESTADOS vs ÁRVORE DE BUSCA:
  ► O espaço de estados é o GRAFO de todos os